<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/LLM/VLM_visionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U transformers accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 33.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [3]:
from huggingface_hub import snapshot_download

# Choose your target path on Google Drive
save_directory = "/content/drive/MyDrive/returning_back/LLM/Qwen2/Qwen2-VL-7B-Instruct/"
model_path = save_directory
snapshot_download(
    repo_id="Qwen/Qwen2-VL-7B-Instruct",
    local_dir=save_directory,
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:208: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

'/content/drive/MyDrive/returning_back/LLM/Qwen2/Qwen2-VL-7B-Instruct'

In [6]:
!pip install -U bitsandbytes>=0.46.1

In [12]:
!pip install -U transformers accelerate bitsandbytes qwen-vl-utils

  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached qwen_vl_utils-0.0.14-py3-none-any.whl.metadata (9.0 kB)
  Using cached av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
Using cached qwen_vl_utils-0.0.14-py3-none-any.whl (8.1 kB)
Using cached av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl (35.8 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

!pip install -U bitsandbytes>=0.46.1

# Path where you downloaded the model in the previous step
model_path = "/content/drive/MyDrive/returning_back/LLM/Qwen2/Qwen2-VL-7B-Instruct/"

# Configure 4-bit quantization for T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load the processor and quantized model
processor = AutoProcessor.from_pretrained(model_path)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="auto"
)


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [5]:
from qwen_vl_utils import process_vision_info
import torch # Added import for torch.cuda.empty_cache()

torch.cuda.empty_cache() # Clear CUDA cache to free up memory

# Prepare your conversation layout
img_path = "/content/drive/MyDrive/test_img.png"
video_path = "/content/drive/MyDrive/test_video.mp4"
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img_path, "max_pixels": 1024 * 768}, # Added max_pixels to reduce image resolution
            {"type": "text", "text": "Describe what you see in this image."}
        ],
    }
]

# Process inputs for the model
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

# Generate response
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    print(output_text[0])

In the image, there is a woman and two children sitting together. The woman is wearing a gray tank top and appears to be holding the children. The children are wearing pajamas, with one child in a pink and white outfit and the other in a red outfit. The setting appears to be a cozy indoor environment with large windows that allow natural light to enter. Outside the window, there is a view of a green landscape with trees and a building. The overall atmosphere seems relaxed and comfortable.


In [11]:
import torch
from qwen_vl_utils import process_vision_info
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache() # Clear CUDA cache to free up memory

# -------------------------------------------------------------
# Define paths to your local files inside your Google Drive
# -------------------------------------------------------------
local_image_path = img_path#"/content/drive/MyDrive/path_to_your_folder/sample.jpg"
local_video_path = video_path#"/content/drive/MyDrive/path_to_your_folder/sample.mp4"

# -------------------------------------------------------------
# Multi-modal Conversation Layout (Images & Videos)
# -------------------------------------------------------------
messages = [
    {
        "role": "user",
        "content": [
            # 1. Processing a local image from Google Drive
            {"type": "image", "image": local_image_path, "max_pixels": 768 * 768}, # Further reduced image resolution

            # 2. Processing a local video from Google Drive
            {
                "type": "video",
                "video": local_video_path,
                # Optional parameters to restrict VRAM consumption on T4:
                "fps": 1.0,           # Extract 1 frame per second of video
                "max_pixels": 256*256 # Further reduced video resolution
            },

            # 3. Text prompt referring to both assets
            {"type": "text", "text": "Describe the local image and summarize the main action happening in the video."}
        ],
    }
]
# -------------------------------------------------------------
# Tokenization & Input Processing
# -------------------------------------------------------------
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")
# -------------------------------------------------------------
# Generate Response
# -------------------------------------------------------------
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    print("Model Output:\n", output_text[0])

AssertionError: The max_pixels of image must be greater than or equal to min_pixels.

In [ ]:
pip install -U langgraph

In [12]:
from typing import Dict, List, Any, TypedDict, Optional
from langgraph.graph import StateGraph, END
import torch
from qwen_vl_utils import process_vision_info
# =====================================================================
# 1. DEFINE THE GRAPH STATE
# =====================================================================
class VisionState(TypedDict):
    """The state object passed between nodes in the graph."""
    image_path: Optional[str]      # Path to local image in Google Drive
    video_path: Optional[str]      # Path to local video in Google Drive
    prompt: str                    # The text instructions for the model
    response: Optional[str]        # Final text generated by Qwen2-VL
    vram_protection: bool          # Toggle low-res optimization flags
# =====================================================================
# 2. CREATE THE QWEN2-VL NODE FUNCTION
# =====================================================================
def qwen_vision_node(state: VisionState) -> Dict[str, Any]:
    """
    LangGraph node that consumes the current state, constructs a multi-modal
    payload, and safely performs quantized inference.
    """
    # Build content list dynamically based on what assets are provided in state
    content_list = []

    if state.get("image_path"):
        content_list.append({"type": "image", "image": state["image_path"]})

    if state.get("video_path"):
        video_payload = {"type": "video", "video": state["video_path"]}
        # Apply strict parameters for T4 VRAM protection if toggled
        if state.get("vram_protection", True):
            video_payload["fps"] = 1.0
            video_payload["max_pixels"] = 360 * 360
        content_list.append(video_payload)

    # Append the core text prompt instructions
    content_list.append({"type": "text", "text": state["prompt"]})
    # Structure the message history standard for Qwen2-VL templates
    messages = [{"role": "user", "content": content_list}]

    # -------------------------------------------------------------
    # Tokenization & Inference Processing
    # -------------------------------------------------------------
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=256)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

    # Return the dictionary to update the graph state keys
    return {"response": output_text[0]}


In [13]:

# =====================================================================
# 3. BUILD AND COMPILE THE GRAPH
# =====================================================================
# Initialize graph with the state schema
workflow = StateGraph(VisionState)

# Add our inference worker node
workflow.add_node("qwen_processor", qwen_vision_node)

# Set the flow topology (Direct Entry -> Processing Node -> Graph Finish)
workflow.set_entry_point("qwen_processor")
workflow.add_edge("qwen_processor", END)

# Compile into an executable application runnable item
app = workflow.compile()

In [14]:
# Configure your initial parameters
initial_inputs = {
    "image_path": img_path,
    "video_path": video_path,
    "prompt": "Extract the total price from the image and describe the incident in the video.",
    "vram_protection": True
}

# Run the graph
final_state = app.invoke(initial_inputs)

# Output results retrieved from graph state dictionary
print("\n--- Pipeline Execution Output ---")
print(final_state["response"])

OutOfMemoryError: CUDA out of memory. Tried to allocate 223.28 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.05 GiB is free. Including non-PyTorch memory, this process has 10.51 GiB memory in use. Of the allocated memory 10.33 GiB is allocated by PyTorch, and 52.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)